# Conferindo o aprendizado

Treina uma política compartilhada no mundo emergente e inspeciona:

1. quais estados foram visitados;
2. a distribuição de ações escolhidas;
3. a Q-table aprendida;
4. uma trajetória narrada com eventos.

In [ ]:
import random
from collections import Counter

import numpy as np

from immune_rl.dynamics import ImmuneHomeostasisReward
from immune_rl.emergent import EmergentDynamics, build_seed_graph
from immune_rl.encoding import EmergentStateEncoder
from immune_rl.entities import Action
from immune_rl.environment import ImmuneEnvironment
from immune_rl.policies import QLearningPolicy

In [ ]:
TRAIN_EPISODES = 300
EVAL_EPISODES = 20
SEED = 0
MAX_STEPS = 40

LEVELS = ("baixo", "médio", "alto")
PHENOTYPES = ("M0", "M1", "M2")
NAMES = ("fenótipo", "ativação", "PAMP", "P", "pró", "anti", "quimio", "reparo", "dano")

In [ ]:
def make_env(seed=SEED):
    return ImmuneEnvironment(
        build_seed_graph,
        EmergentDynamics(seed=seed),
        ImmuneHomeostasisReward(),
        max_steps=MAX_STEPS,
    )


def train_shared(env, policy):
    for _ in range(TRAIN_EPISODES):
        obs = env.reset()
        while True:
            actions = {cid: policy.select_action(o) for cid, o in obs.items()}
            next_obs, rewards, terminated, truncated, _ = env.step(actions)
            for cid in obs:
                if cid in next_obs and cid in rewards:
                    policy.update(
                        obs[cid], actions[cid], rewards[cid], next_obs[cid], terminated
                    )
            obs = next_obs
            if terminated or truncated:
                break
        policy.end_episode()


def describe(state):
    parts = []
    for name, value in zip(NAMES, state):
        label = PHENOTYPES[value] if name == "fenótipo" else LEVELS[value]
        parts.append(f"{name}={label}")
    return " | ".join(parts)


def count_visits(env, policy, episodes=EVAL_EPISODES):
    visits = Counter()
    actions = Counter()
    for _ in range(episodes):
        obs = env.reset()
        while True:
            for observation in obs.values():
                visits[policy.encoder.encode(observation)] += 1
            chosen = {cid: policy.select_action(o) for cid, o in obs.items()}
            actions.update(action.name for action in chosen.values())
            obs, _, terminated, truncated, _ = env.step(chosen)
            if terminated or truncated:
                break
    return visits, actions


def print_q_table(policy, visits, top=15):
    print(f"{'visitas':>8}  {'estado':<62}{'melhor ação':<14}Q")
    for state, count in visits.most_common(top):
        q = policy.q[state]
        best = Action(int(np.argmax(q)))
        print(f"{count:>8}  {describe(state):<62}{best.name:<14}{q.max():+.3f}")


def print_trajectory(env, policy, steps=MAX_STEPS):
    obs = env.reset()
    for t in range(steps):
        actions = {cid: policy.select_action(o) for cid, o in obs.items()}
        obs, rewards, terminated, truncated, info = env.step(actions)
        reward = sum(rewards.values()) / max(len(rewards), 1)
        acts = ", ".join(f"{cid}:{action.name}" for cid, action in actions.items())
        messages = "; ".join(event.message for event in info["events"])
        print(
            f"t={t + 1:02d} R={reward:+.3f} P={info['P']:.2f} pró={info['Pro']:.2f} "
            f"dano={info['D']:.2f} M1={info['M1']} M2={info['M2']}"
        )
        print(f"   ações: {acts}")
        print(f"   eventos: {messages or 'nenhum'}")
        if terminated or truncated:
            print("resultado:", info["outcome"])
            break

In [ ]:
random.seed(SEED)
env = make_env()
shared = QLearningPolicy(EmergentStateEncoder(), seed=SEED)
train_shared(env, shared)
shared.epsilon = 0.0
print(f"Q-table: {len(shared.q)} estados visitados")

In [ ]:
visits, chosen = count_visits(env, shared)
total = sum(chosen.values())
for action, count in chosen.most_common():
    print(f"{action:<14}{count:>6} ({count / total:5.1%})")

In [ ]:
print_q_table(shared, visits)

In [ ]:
print_trajectory(make_env(), shared)

## O que observar

- Estados com M0 costumam aparecer no início; M1/M2 aparecem depois que a
  secreção altera os escores de fenótipo.
- `SECRETE_PRO` deve dominar enquanto há patógeno; `SECRETE_REPAIR` aparece
  quando o dano sobe.
- A trajetória mostra a transição de resposta inflamatória para reparo e o
  encerramento apenas quando patógeno e dano estão baixos.

Se a política ficar presa em uma ação, aumente `TRAIN_EPISODES` ou ajuste os
pesos de `ImmuneHomeostasisReward`.